Bible verse creator

In [1]:
# imports

import os
import json
import sys
sys.path.append("../..")
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [10]:
openai = OpenAI()
MODEL = "gpt-5"

system_prompt = "You generate encouraging bible verses (maximum 3) for the user based on their input. You are a helpful assistant that provides relevant bible verses to the user. Ensure to return you response in JSON format.The JSON format should be as follows: { 'verses': [ { 'verse': 'John 3:16', 'text': 'For God so loved the world that he gave his one and only Son, that whoever believes in him shall not perish but have eternal life.', 'translation': 'NIV' },  { 'verse': 'Psalm 23:4', 'text': 'Even though I walk through the darkest valley, I will fear no evil, for you are with me; your rod and your staff, they comfort me.', 'translation': 'NIV' } ] }"


In [5]:
# Bible verses
def select_verses(user_prompt):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    verses = json.loads(result)
    print(f"Found {len(verses['verses'])} relevant verses")
    return verses

In [ ]:
# Select verses based on user input
verses = select_verses("I am feeling sad and need some encouragement.")
print (verses)  

Found 3 relevant verses
{'verses': [{'verse': 'Psalm 34:18', 'text': 'The LORD is close to the brokenhearted and saves those who are crushed in spirit.', 'translation': 'NIV'}, {'verse': 'Isaiah 41:10', 'text': 'So do not fear, for I am with you; do not be dismayed, for I am your God. I will strengthen you and help you; I will uphold you with my righteous right hand.', 'translation': 'NIV'}, {'verse': 'Matthew 11:28', 'text': 'Come to me, all you who are weary and burdened, and I will give you rest.', 'translation': 'NIV'}]}


In [15]:
#create a function to display the verses in a nice format       
def display_verses(verses):
    for verse in verses['verses']:
        display(Markdown(f"**{verse['verse']} {verse['text']}** - {verse['translation']}"))     

In [16]:
# Display the selected verses
display_verses(verses)      

**Psalm 34:18 The LORD is close to the brokenhearted and saves those who are crushed in spirit.** - NIV

**Isaiah 41:10 So do not fear, for I am with you; do not be dismayed, for I am your God. I will strengthen you and help you; I will uphold you with my righteous right hand.** - NIV

**Matthew 11:28 Come to me, all you who are weary and burdened, and I will give you rest.** - NIV

In [ ]:
# Devotional Content
def generate_devotional_content(verses):
    devotional_prompt = f"Create a short devotional based on the following bible verses: {json.dumps(verses)}"
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that generates devotional content based on provided bible verses. Ensure to return your response in JSON format with the following structure: { 'devotional': 'First paragraph.\\n\\nSecond paragraph.' }. The devotional must be exactly two paragraphs separated by a blank line (\\n\\n)."},
            {"role": "user", "content": devotional_prompt}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    devotional_content = json.loads(result)
    return devotional_content

In [20]:
# Output the devotional content
generated_devotional_content = generate_devotional_content(verses)
print(generated_devotional_content)

{'devotional': 'When your heart aches and your strength feels spent, remember: the Lord draws nearest in the very place of your pain (Psalm 34:18). He is not a distant observer but your God, speaking courage into your fear and placing His righteous right hand beneath your weakness (Isaiah 41:10). Jesus invites you to come—just as you are, weary and burdened—and He will give you rest (Matthew 11:28). Today, exhale the weight you carry and inhale His presence; trade panic for prayer, hurry for His yoke, and despair for the hope of being held. Whisper, “Lord, I come to You. Hold my heart, steady my steps, and give me Your rest.”'}


In [28]:
# Display the devotional content
display(Markdown(f"**Devotional Content:** {generated_devotional_content['devotional']}"))

**Devotional Content:** When your heart aches and your strength feels spent, remember: the Lord draws nearest in the very place of your pain (Psalm 34:18). He is not a distant observer but your God, speaking courage into your fear and placing His righteous right hand beneath your weakness (Isaiah 41:10). Jesus invites you to come—just as you are, weary and burdened—and He will give you rest (Matthew 11:28). Today, exhale the weight you carry and inhale His presence; trade panic for prayer, hurry for His yoke, and despair for the hope of being held. Whisper, “Lord, I come to You. Hold my heart, steady my steps, and give me Your rest.”

In [29]:
import re

ABBREVIATIONS = {"St.", "Mr.", "Mrs.", "Ms.", "Dr.", "Jr.", "Sr.", "Rev.", "vs.", "etc.", "e.g.", "i.e."}

def split_sentences(text):
    """Split text into sentences, not treating known abbreviations (e.g. St.) as sentence ends."""
    sentences = []
    for part in re.split(r"(?<=[.!?])\s+", text.strip()):
        if sentences and sentences[-1].split()[-1] in ABBREVIATIONS:
            sentences[-1] += " " + part
        else:
            sentences.append(part)
    return sentences

def to_paragraphs(text, sentences_per_paragraph=2):
    """Group every N sentences into a paragraph, separated by blank lines."""
    sentences = split_sentences(text)
    groups = [sentences[i:i + sentences_per_paragraph] for i in range(0, len(sentences), sentences_per_paragraph)]
    return "\n\n".join(" ".join(group) for group in groups)

def display_json_markdown(output, sentences_per_paragraph=2):
    """Display each key of a JSON dict as a heading, with its text split into paragraphs."""
    for key, value in output.items():
        heading = key.replace("_", " ").title()
        display(Markdown(f"## {heading}\n\n{to_paragraphs(str(value), sentences_per_paragraph)}"))

display_json_markdown(generated_devotional_content)

## Devotional

When your heart aches and your strength feels spent, remember: the Lord draws nearest in the very place of your pain (Psalm 34:18). He is not a distant observer but your God, speaking courage into your fear and placing His righteous right hand beneath your weakness (Isaiah 41:10).

Jesus invites you to come—just as you are, weary and burdened—and He will give you rest (Matthew 11:28). Today, exhale the weight you carry and inhale His presence; trade panic for prayer, hurry for His yoke, and despair for the hope of being held.

Whisper, “Lord, I come to You. Hold my heart, steady my steps, and give me Your rest.”